# Decision Policy Demo

This notebook will provide code demonstrating how to use Decision Policies as introduced in Wait! There's a Way Out. This notebook will also provide code for running the experiments in the paper. 

In [1]:
# TODO

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

In [2]:
import os
import argparse
import sys
import glob

from functools import partial
import json
from convokit import Corpus, Forecaster, download
from convokit.forecaster.TransformerDecoderModel import TransformerDecoderModel
from convokit.forecaster.TransformerForecasterConfig import TransformerForecasterConfig
from convokit.decisionpolicy import DeferralDecisionPolicy, ThresholdDecisionPolicy
from convokit.utterance_simulator.unslothUtteranceSimulatorModel import UnslothUtteranceSimulatorModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-28 07:19:59.106329: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 07:19:59.126464: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777360799.150730 1126746 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777360799.158776 1126746 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777360799.179235 1126746 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Set repo root

from pathlib import Path

repo_root = Path.cwd()
while repo_root.name != "ConvoKit":
    repo_root = repo_root.parent

repo_root = str(repo_root)

In [ ]:
# TODO temporary pre-merge downloader for decisionpolicy-demo
from pathlib import Path
import json
import urllib.request
import zipfile

from convokit import Corpus

DOWNLOAD_CONFIG_URL = (
    "https://raw.githubusercontent.com/laerdon/ConvoKit/"
    "master/download_config.json"
)


def get_decisionpolicy_demo_url(config_url=DOWNLOAD_CONFIG_URL):
    print(f"[info] reading download config from {config_url}")
    with urllib.request.urlopen(config_url) as response:
        dataset_config = json.load(response)
    try:
        return dataset_config["DatasetURLs"]["decisionpolicy-demo"]
    except KeyError as exc:
        raise KeyError(
            "decisionpolicy-demo is missing from laerdon/ConvoKit master download_config.json"
        ) from exc


def download_decisionpolicy_demo(data_dir=None):
    data_root = Path(data_dir or "~/.convokit/saved-corpora").expanduser()
    dataset_name = "decisionpolicy-demo"
    dataset_dir = data_root / dataset_name
    zip_path = data_root / f"{dataset_name}.zip"

    if any(path.is_dir() and (path / "index.json").exists() for path in dataset_dir.rglob("*")):
        print(f"[info] using cached {dataset_name} at {dataset_dir}")
        return dataset_dir

    url = get_decisionpolicy_demo_url()
    data_root.mkdir(parents=True, exist_ok=True)
    print(f"[info] downloading {dataset_name} from {url}")
    urllib.request.urlretrieve(url, zip_path)

    print(f"[info] extracting {zip_path} to {data_root}")
    with zipfile.ZipFile(zip_path, "r") as zipf:
        zipf.extractall(data_root)

    if not dataset_dir.exists():
        raise FileNotFoundError(f"expected extracted folder missing: {dataset_dir}")
    return dataset_dir

base = download_decisionpolicy_demo()

[info] using cached decisionpolicy-demo at /home/lyk25/.convokit/saved-corpora/decisionpolicy-demo
[info] loaded 26 corpora from /home/lyk25/.convokit/saved-corpora/decisionpolicy-demo


In [6]:
import re
corpus_dirs = sorted(
    corpus_dir
    for corpus_dir in base.rglob("*")
    if corpus_dir.is_dir() and (corpus_dir / "index.json").exists()
)
if not corpus_dirs:
    raise FileNotFoundError(f"no convokit corpora found under {base}")
_seed_policy_re = re.compile(r"^seed-(\d+)-(.+)$")
_test_seed_re = re.compile(r"^test-seed-(\d+)$")
corpora_all = {}
corpora = {}
for corpus_dir in corpus_dirs:
    if _test_seed_re.match(corpus_dir.name):
        corpora[corpus_dir.name] = Corpus(filename=str(corpus_dir))
        continue
    m_test = _test_seed_re.match(corpus_dir.parent.name)
    if m_test:
        corpora[f"test-seed-{m_test.group(1)}"] = Corpus(filename=str(corpus_dir))
        continue
    m = _seed_policy_re.match(corpus_dir.parent.name)
    if m and corpus_dir.name == m.group(2):
        corpora_all[f"seed-{m.group(1)}-{m.group(2)}"] = Corpus(filename=str(corpus_dir))
CORPUS_POLICY_PER_SEED = "DeferralDecisionPolicy"
if not corpora:
    for key in sorted(corpora_all):
        m = _seed_policy_re.match(key)
        if m and m.group(2) == CORPUS_POLICY_PER_SEED:
            corpora[f"test-seed-{m.group(1)}"] = corpora_all[key]
if not corpora:
    raise FileNotFoundError(
        f"no test-seed-<n> corpora under {base} and none derived from {CORPUS_POLICY_PER_SEED!r}"
    )
print(
    f"[info] test-seed corpora: {len(corpora)} ({', '.join(sorted(corpora))}); "
    f"seed-* policy corpora in corpora_all: {len(corpora_all)}"
)

[info] using cached decisionpolicy-demo at /home/lyk25/.convokit/saved-corpora/decisionpolicy-demo
[info] test-seed corpora: 5 (test-seed-1, test-seed-2, test-seed-3, test-seed-4, test-seed-5); seed-* policy corpora in corpora_all: 25


In [8]:
corpora_all

{'seed-1-DeferralDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f14bdbf4cd0>,
 'seed-1-RandomDeferralDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f14bd2da1d0>,
 'seed-1-SimulationAverageDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10f75d3f50>,
 'seed-1-SimulationMajorityDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10ef43ffd0>,
 'seed-1-ThresholdDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10e74a7fd0>,
 'seed-2-DeferralDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f1117bd0190>,
 'seed-2-RandomDeferralDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f1122031050>,
 'seed-2-SimulationAverageDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f110d88bf90>,
 'seed-2-SimulationMajorityDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10dff13f90>,
 'seed-2-ThresholdDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10d85a3f50>,
 'seed-3-DeferralDecisionPolicy': <convokit.model.corpus.Corpus at 0x7f10d0b33fd0>,
 'seed-3-RandomDeferralD

Having imported our DeferralDecisionPolicy and ThresholdDecisionPolicy, we now will first define all of the other decision policies to benchmark. 

In [ ]:
from typing import Callable, List, Optional, Dict, Any, Tuple
import numpy as np
from convokit.decisionpolicy import DecisionPolicy


class _synthetic_speaker:
    def __init__(self, speaker_id: str):
        self.id = speaker_id


class _synthetic_utterance:
    def __init__(self, text: str, utterance_id: str, speaker_id: str):
        self.text = text
        self.id = utterance_id
        self.speaker_ = _synthetic_speaker(speaker_id)
        self.meta = {}

    def get_conversation(self):
        return None


class RandomDeferralDecisionPolicy(DecisionPolicy):
    """
    Decision policy that defers intervention by looking ahead at simulated next utterances.

    :param simulator: utterance simulator model (must have a ``transform(contexts)`` method
        returning a DataFrame indexed by utterance id). if the simulator exposes
        ``get_num_simulations()``, ``num_simulations`` is capped to that value.
    :param threshold: probability threshold above which a context is flagged.
    :param tau: minimum number of simulated branches that must exceed the threshold
        before an intervention is issued.
    :param num_simulations: how many simulated branches to use per context (capped to
        simulator's ``get_num_simulations()`` if available).
    :param store_simulations: if True, simulated reply strings are cached during decide()
        and written to corpus utterance metadata by post_transform().
    :param simulated_reply_attribute_name: metadata field name used when storing simulations
        on corpus utterances (only relevant when store_simulations=True).
    """

    def __init__(
        self,
        simulator,
        threshold,
        deferral_probability: float = 0.1515,
        reuse_cached_forecast_probs: bool = True,
        forecast_prob_attribute_name: str = "forecast_prob",
    ):
        # forward the cache flag to the base class so its _score helper honors it.
        # without this, reuse_cached_probabilities on this subclass had no effect.
        super().__init__(
            forecast_prob_attribute_name=forecast_prob_attribute_name,
            reuse_cached_forecast_probs=reuse_cached_forecast_probs,
        )
        self.simulator = simulator
        self.threshold = float(threshold)
        self.deferral_probability = float(deferral_probability)

    def _decision_score(self, context, score_fn: Callable):
        # use base _score so a cached forecast_prob on the utterance meta is reused
        # instead of re-invoking the belief estimator.
        return self._score(context, score_fn)

    def decide(self, context, score_fn: Callable) -> Tuple[float, int, Optional[Dict[str, Any]]]:
        decision_score = self._score(context, score_fn)

        p = float(np.random.rand())
   

        # return an empty metadata dict (not None) so downstream code that iterates
        # utt_metadata.items() in TransformerDecoderModel.transform doesn't crash.
        return (decision_score,
        1 if decision_score > self.threshold and p > self.deferral_probability else 0,
        {}
        )

    def fit(self, contexts, val_contexts=None, score_fn: Callable = None):
        if val_contexts is None or score_fn is None or self.labeler is None:
            print("either no validation contexts/score function/labeler were provided, returning current threshold")
            return {"best_threshold": self.threshold}

        val_contexts = list(val_contexts)
        if len(val_contexts) == 0:
            print("no validation contexts were provided, returning current threshold")
            return {"best_threshold": self.threshold}

        fit_result = self._fit_with_model_checkpoint_selection(val_contexts, score_fn=score_fn)
        if isinstance(fit_result, dict):
            if "best_threshold" in fit_result:
                self.threshold = float(fit_result["best_threshold"])
            return fit_result

        fit_result = self._fit_threshold_for_loaded_model(val_contexts, score_fn=score_fn)
        if "best_threshold" in fit_result:
            self.threshold = float(fit_result["best_threshold"])
        return fit_result

In [ ]:
from typing import Callable, Optional, Dict, Any, Tuple

from convokit.decisionpolicy import DeferralDecisionPolicy


class SimulationAverageDecisionPolicy(DeferralDecisionPolicy):
    """
    decision policy that intervenes if the mean of the simulated next-utterance
    scores is at or above the threshold.

    this subclass inherits all simulation fetching, per-utterance metadata
    caching, sim-score caching, and threshold fitting from
    DeferralDecisionPolicy. the only differences are:
      * no ``tau`` parameter (unused; forwarded as 0 to super)
      * ``decide`` predicts based on mean(simulation_scores) >= threshold
    """

    def __init__(
        self,
        simulator,
        threshold,
        num_simulations: int = 10,
        store_simulations: bool = False,
        simulated_reply_attribute_name: str = "sim_replies",
        sim_replies_forecast_probs_attribute_name: str = "sim_replies_forecast_probs",
        reuse_cached_simulations: bool = True,
    ):
        # tau is irrelevant for the mean-based decision rule, so we pin it to 0
        # upstream rather than expose it to callers of this subclass.
        super().__init__(
            simulator=simulator,
            threshold=threshold,
            tau=0,
            num_simulations=num_simulations,
            store_simulations=store_simulations,
            simulated_reply_attribute_name=simulated_reply_attribute_name,
            sim_replies_forecast_probs_attribute_name=sim_replies_forecast_probs_attribute_name,
            reuse_cached_simulations=reuse_cached_simulations,
        )

    def decide(self, context, score_fn: Callable) -> Tuple[float, int, Optional[Dict[str, Any]]]:
        decision_score, simulations, simulation_scores = self._decision_score(context, score_fn)
        # empty simulation_scores would zero-divide. this happens when the
        # simulator returns no completions for a context (e.g. end-of-conversation
        # contexts that slip through the selector). treat as no intervention so
        # a single degenerate context doesn't abort the whole transform run.
        if len(simulation_scores) == 0:
            average_simulation_score = 0.0
            pred = 0
        else:
            average_simulation_score = sum(simulation_scores) / len(simulation_scores)
            pred = 1 if average_simulation_score >= self.threshold else 0
        return (
            decision_score,
            pred,
            {
                self.simulated_reply_attribute_name: simulations,
                self.sim_replies_forecast_probs_attribute_name: simulation_scores,
            },
        )


In [ ]:
from typing import Callable, Optional, Dict, Any, Tuple

from convokit.decisionpolicy import DeferralDecisionPolicy


class SimulationMajorityDecisionPolicy(DeferralDecisionPolicy):
    """
    decision policy that intervenes if at least ``tau`` of the simulated next
    utterances score above the threshold, ignoring the current utterance score.

    this subclass inherits all simulation fetching, per-utterance metadata
    caching, sim-score caching, and threshold fitting from
    DeferralDecisionPolicy. the only difference is in ``decide``: the gate
    ``decision_score > threshold`` is dropped so that only the simulated-branch
    vote count drives the prediction.
    """

    def decide(self, context, score_fn: Callable) -> Tuple[float, int, Optional[Dict[str, Any]]]:
        decision_score, simulations, simulation_scores = self._decision_score(context, score_fn)
        num_simulations_above_threshold = sum(
            1 for score in simulation_scores if score > self.threshold
        )
        return (
            decision_score,
            1 if num_simulations_above_threshold >= self.tau else 0,
            {
                self.simulated_reply_attribute_name: simulations,
                self.sim_replies_forecast_probs_attribute_name: simulation_scores,
            },
        )


Since we use simulations in our baselines, we must define a simulation configuration as follows: 

In [ ]:
SIMULATOR_TRAIN_CONFIG = {
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "eval_strategy": "steps",
    "save_strategy": "steps",
    "save_steps": 30,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 5,
    "num_train_epochs": 1,
    "eval_steps": 30,
    "learning_rate": 2e-4,
    "logging_steps": 5,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "output_dir": "outputs/simulator_finetune",
    "logging_dir": "logs",
    "load_best_model_at_end": True,
}

In [ ]:
TAU = 7
DEFERRAL_PROBABILITY_THRESHOLD = 0.2518938553561718
NUM_SIMULATIONS = 10
OUTPUT_DIR = "benchmark_preannotated"
SEEDS = [1,2,3,4,5]

In [ ]:
simulator_model = UnslothUtteranceSimulatorModel(
    model_name="/reef/lyk25/dynamic_training/game_analysis/outputs/checkpoint-74",
    train_config=SIMULATOR_TRAIN_CONFIG,
)

==((====))==  Unsloth 2025.7.11: Fast Llama patching. Transformers: 4.53.3.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.536 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.1.0+cf34004b8a
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.7.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


After loading the simulator model, we define context selectors.

In [13]:
def context_selector(context_tuple, split):
    """
    We use this generic function for both training and validation data.
    In both cases, its job is to select only those contexts for which the
    FUTURE context is not empty, so we have a next utterance to predict.
    """
    matches_split = (context_tuple.current_utterance.get_conversation().meta.get("split") == split)
    is_end = (len(context_tuple.future_context) == 0)
    return matches_split and not is_end

def make_data_selector(split):
    return lambda context_tuple: context_tuple.current_utterance.get_conversation().meta.get("split") == split

train_context_selector = partial(context_selector, split="train")
val_context_selector = partial(context_selector, split="val")
test_context_selector = partial(context_selector, split="test")

Below is a script to fully reproduce, sans regenerating simulations (simulations requires substantially more compute).

In [ ]:
for seed_idx in SEEDS:
    train_corpus = Corpus(filename=download('conversations-gone-awry-cmv-corpus-large'))
    # corpus = Corpus(filename=f'/reef/lyk25/theres-a-way-out-ACL26-internal/outputs/benchmark_preannotated/seed-{seed_idx}-ThresholdDecisionPolicy/ThresholdDecisionPolicy')
    # corpus = Corpus(filename=f'/reef/lyk25/dynamic_training/game_analysis/corpi/test/test-son-seed-{seed_idx}')
    corpus = Corpus(filename=download('conversations-gone-awry-cmv-corpus-large'))
    corpus.filter_conversations_by(lambda convo: convo.meta['split'] == 'test')

    config = TransformerForecasterConfig(
        output_dir=f"outputs/{OUTPUT_DIR}/forecaster_{seed_idx}",
        per_device_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        learning_rate=1e-5,
        random_seed=seed_idx,
        context_mode="normal",
        device="cuda",
    )

    # TODO this will have to be edited
    forecaster_model = TransformerDecoderModel(
        model_name_or_path="google/gemma-2-9b-it",
        config=config,
    )

    forecaster = Forecaster(
        forecaster_model=forecaster_model,
        labeler='has_removed_comment',
    )

    forecaster.fit_belief_estimator(
        corpus=train_corpus,
        context_selector=train_context_selector,
        val_context_selector=val_context_selector,
    )

    # ---
    cfg_path = os.path.join(repo_root, "saves", f"seed-{seed_idx}", "dev_config.json")
    with open(cfg_path) as f:
        cfg = json.load(f)
    best_threshold = cfg['best_threshold']

    for policy_trial in [ThresholdDecisionPolicy, DeferralDecisionPolicy, RandomDeferralDecisionPolicy, SimulationAverageDecisionPolicy, SimulationMajorityDecisionPolicy]:
        print('---')
        print(f"Fitting policy {policy_trial.__name__} for seed {seed_idx}")
        if policy_trial == ThresholdDecisionPolicy:
            policy = ThresholdDecisionPolicy(
                threshold=best_threshold,
                reuse_cached_forecast_probs=False,
            )
        elif policy_trial == DeferralDecisionPolicy:
            policy = DeferralDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                tau=TAU,
                reuse_cached_forecast_probs=False,
            )
        elif policy_trial == RandomDeferralDecisionPolicy:
            policy = RandomDeferralDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                deferral_probability=DEFERRAL_PROBABILITY_THRESHOLD,
                reuse_cached_forecast_probs=False,
            )
        elif policy_trial == SimulationAverageDecisionPolicy:
            policy = SimulationAverageDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                num_simulations=NUM_SIMULATIONS,
                store_simulations=False,
                simulated_reply_attribute_name="sim_replies",
                sim_replies_forecast_probs_attribute_name="sim_replies_forecast_probs",
                reuse_cached_forecast_probs=False,
            )
        elif policy_trial == SimulationMajorityDecisionPolicy:
            policy = SimulationMajorityDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                tau=TAU,
                num_simulations=NUM_SIMULATIONS,
                store_simulations=False,
                simulated_reply_attribute_name="sim_replies",
                sim_replies_forecast_probs_attribute_name="sim_replies_forecast_probs",
                reuse_cached_forecast_probs=False,
            )
        
        # attach the decision policy to the underlying forecaster model;
        # Forecaster itself does not accept decision_policy in its constructor.
        forecaster_model.decision_policy = policy

        forecaster = Forecaster(
            forecaster_model=forecaster_model,
            labeler='has_removed_comment',
        )

        print('starting transformation.')
        # evaluate the forecaster on the test set
        forecaster.transform(
            corpus=corpus,
            context_selector=make_data_selector('test'),
            verbose=True,
        )
        print('transformation complete.')

        output_dir = f"outputs/{OUTPUT_DIR}/seed-{seed_idx}-{policy_trial.__name__}"
        os.makedirs(output_dir, exist_ok=True)
        corpus.dump(name=f"{policy_trial.__name__}", base_path=output_dir)
        print('corpus dumped.')

        print('starting summarization.')
        # forecaster.summarize expects a conversation-level selector (Callable[[Conversation], bool]),
        # unlike the context-tuple selectors used in fit/transform.
        def summarize_selector(convo):
            return convo.meta.get("split") == "test"
        conversational_forecasts_df, metrics = forecaster.summarize(
            corpus=corpus,
            selector=summarize_selector,
        )
        print('summarization complete.')
        
        # path to the seed output directory
        seed_folder = f"outputs/{OUTPUT_DIR}/seed-{seed_idx}-{policy_trial.__name__}"

        # ensure the directory exists
        os.makedirs(seed_folder, exist_ok=True)

        # save conversational_forecasts_df as CSV
        conversational_forecasts_df.to_csv(os.path.join(seed_folder, "conversational_forecasts.csv"), index=False)

        # save metrics as JSON
        with open(os.path.join(seed_folder, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

A faster reproduction is possible by skipping the training and transformation.

In [ ]:
for seed_idx in SEEDS:
    config = TransformerForecasterConfig(
        output_dir=f"outputs/{OUTPUT_DIR}/forecaster_{seed_idx}",
        per_device_batch_size=16,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        learning_rate=1e-5,
        random_seed=seed_idx,
        context_mode="normal",
        device="cuda",
    )

    # TODO this will have to be edited
    forecaster_model = TransformerDecoderModel(
        model_name_or_path="google/gemma-2-9b-it",
        config=config,
    )

    forecaster = Forecaster(
        forecaster_model=forecaster_model,
        labeler='has_removed_comment',
    )

    # remove training---we can instead use the cached forecast probabilities.

    # ---
    cfg_path = os.path.join(repo_root, "saves", f"seed-{seed_idx}", "dev_config.json")
    with open(cfg_path) as f:
        cfg = json.load(f)
    best_threshold = cfg['best_threshold']

    for policy_trial in [ThresholdDecisionPolicy, DeferralDecisionPolicy, RandomDeferralDecisionPolicy, SimulationAverageDecisionPolicy, SimulationMajorityDecisionPolicy]:
        corpus_name = f"seed-{seed_idx}-{policy_trial.__name__}"
        if corpus_name in corpora:
            corpus = corpora_all[corpus_name]
        else:
            raise KeyError(f"missing corpus {corpus_name}")

        print('---')
        print(f"fitting policy {policy_trial.__name__} for seed {seed_idx}")
        if policy_trial == ThresholdDecisionPolicy:
            policy = ThresholdDecisionPolicy(
                threshold=best_threshold,
            )
        elif policy_trial == DeferralDecisionPolicy:
            policy = DeferralDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                tau=TAU,
            )
        elif policy_trial == RandomDeferralDecisionPolicy:
            policy = RandomDeferralDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                deferral_probability=DEFERRAL_PROBABILITY_THRESHOLD,
            )
        elif policy_trial == SimulationAverageDecisionPolicy:
            policy = SimulationAverageDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                num_simulations=NUM_SIMULATIONS,
                store_simulations=False,
                simulated_reply_attribute_name="sim_replies",
                sim_replies_forecast_probs_attribute_name="sim_replies_forecast_probs",
            )
        elif policy_trial == SimulationMajorityDecisionPolicy:
            policy = SimulationMajorityDecisionPolicy(
                simulator=simulator_model,
                threshold=best_threshold,
                tau=TAU,
                num_simulations=NUM_SIMULATIONS,
                store_simulations=False,
                simulated_reply_attribute_name="sim_replies",
                sim_replies_forecast_probs_attribute_name="sim_replies_forecast_probs",
            )
        
        # attach the decision policy to the underlying forecaster model;
        # Forecaster itself does not accept decision_policy in its constructor.
        forecaster_model.decision_policy = policy

        forecaster = Forecaster(
            forecaster_model=forecaster_model,
            labeler='has_removed_comment',
        )

        print('starting transformation.')
        # evaluate the forecaster on the test set
        forecaster.transform(
            corpus=corpus,
            context_selector=make_data_selector('test'),
            verbose=True,
        )
        print('transformation complete.')

        output_dir = f"outputs/{OUTPUT_DIR}/seed-{seed_idx}-{policy_trial.__name__}"
        os.makedirs(output_dir, exist_ok=True)
        corpus.dump(name=f"{policy_trial.__name__}", base_path=output_dir)
        print('corpus dumped.')

        print('starting summarization.')
        # forecaster.summarize expects a conversation-level selector (Callable[[Conversation], bool]),
        # unlike the context-tuple selectors used in fit/transform.
        def summarize_selector(convo):
            return convo.meta.get("split") == "test"
        conversational_forecasts_df, metrics = forecaster.summarize(
            corpus=corpus,
            selector=summarize_selector,
        )
        print('summarization complete.')
        
        # path to the seed output directory
        seed_folder = f"outputs/{OUTPUT_DIR}/seed-{seed_idx}-{policy_trial.__name__}"

        # ensure the directory exists
        os.makedirs(seed_folder, exist_ok=True)

        # save conversational_forecasts_df as CSV
        conversational_forecasts_df.to_csv(os.path.join(seed_folder, "conversational_forecasts.csv"), index=False)

        # save metrics as JSON
        with open(os.path.join(seed_folder, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

## Human benchmark analysis

In [12]:
# import human data SQL here

In [13]:
import sqlite3

In [16]:
round_n = 1

In [17]:
# connect to the game database
db_path = '/reef/sqt2/cga-eval/human/game_db.sqlite'
connection = sqlite3.connect(db_path)
cursor = connection.cursor()

print(f"[info] connected to database: {db_path}")
cursor.execute("SELECT COUNT(*) FROM results")

# # names_list = []
# # answers_list = []
# # scores_list = []
# # comments_list = []

# TIME_CUTOFF = 1714059814630  # timestamp to cut off previous results
# TIME_CUTOFF_END = 1730385762530

# for row in connection.execute('SELECT * FROM results'):
#     if row[0] != 'yc2727' and row[0] != 'sqt2':
#         answers = json.loads(row[1])
#         if answers[0]['start_time'] > TIME_CUTOFF and answers[0]['start_time'] < TIME_CUTOFF_END:
#             names_list.append(row[0])
#             answers_list.append(answers)
#             scores_list.append(row[2])
#             comments_list.append(row[3])

if round_n == 1:
    # for round_n 1
    names_list_1 = []
    answers_list_1 = []
    scores_list_1 = []
    comments_list_1 = []

    TIME_CUTOFF_1 = 1730395000000
    TIME_CUTOFF_END_1 = 1730397199000

    for row in connection.execute('SELECT * FROM results'):
        if row[0] != 'yc2727' and row[0] != 'sqt2':
            answers = json.loads(row[1])
            if answers[0]['start_time'] > TIME_CUTOFF_1 and answers[0]['start_time'] < TIME_CUTOFF_END_1:
                names_list_1.append(row[0])
                answers_list_1.append(answers)
                scores_list_1.append(row[2])
                comments_list_1.append(row[3])
elif round_n == 2:
    # for round 2
    names_list_2 = []
    answers_list_2 = []
    scores_list_2 = []
    comments_list_2 = []

    TIME_CUTOFF_2_a = 1731002910000
    TIME_CUTOFF_END_2_a = 1731016180000

    for row in connection.execute('SELECT * FROM results'):
        if row[0] != 'yc2727' and row[0] != 'sqt2':
            answers = json.loads(row[1])
            if answers[0]['start_time'] > TIME_CUTOFF_2_a and answers[0]['start_time'] < TIME_CUTOFF_END_2_a:
                names_list_2.append(row[0])
                answers_list_2.append(answers)
                scores_list_2.append(row[2])
                comments_list_2.append(row[3])

    # The 2_b cutoff below is to specifically add data from 'ljl2' for a later second round window,
    # likely because they submitted their data late or for a different time block.
    TIME_CUTOFF_2_b = 1732212720000
    TIME_CUTOFF_END_2_b = 1740000000000

    for row in connection.execute('SELECT * FROM results'):
        if row[0].lower() == 'ljl2':
            answers = json.loads(row[1])
            if answers[0]['start_time'] > TIME_CUTOFF_2_b and answers[0]['start_time'] < TIME_CUTOFF_END_2_b:
                names_list_2.append(row[0])
                answers_list_2.append(answers)
                scores_list_2.append(row[2])
                comments_list_2.append(row[3])

    # print(f"Round 1: {len(names_list_1)} entries")
    # print(f"Round 2: {len(names_list_2)} entries")
    # names_list_1, names_list_2

[info] connected to database: /reef/sqt2/cga-eval/human/game_db.sqlite


In [18]:
human_map1 = {
    "vn72": ['d3efg03', 'd3f07tv', 'f1353ra', 'f13u0n2', 'fegw71k', 'fegwvia', 'ikpw1ol', 'ikpxahv', 'isz5n6g', 'isz6a7m'],
    "nac86": ['dyd7cfv', 'dydawov', 'ewtjxp2', 'ewtowqu', 'f00nxjs', 'f014doo', 'ii3cpve', 'ii4ivim', 'ikzb571', 'ikzcblg'],
    # "sj597": ['dbnr5nd', 'dbnxbzn', 'g1sm9mt', 'g1spg0k', 'g297nn7', 'g2998qp', 'h2h7yl0', 'h2hauaj', 'ifknq44', 'ifkoazb'],
    "yc2727": ['d08x3kl', 'd08x4q0', 'd3efg03', 'd3f07tv', 'ewtjxp2', 'ewtowqu', 'hnupywh', 'hnuu3ip', 'ih2fn94', 'ih3iwph'],
    # "ex36": ['d0fyu9x', 'd0fzp40', 'dg7wmdb', 'dg7x3eu', 'ej6jusi', 'ej6ollb', 'ewtjxp2', 'ewtowqu', 'f0dryv4', 'f0dsbw4'],
    "kz88": ['doi3vuz', 'doi44j8', 'e9mybxp', 'e9mynuy', 'g1q1973', 'g1q3upb', 'ggwdbsa', 'ggwei5y', 'gmxoyuu', 'gmxqrdz'],
    "LJL2": ['ffpj88q', 'ffpk80c', 'flixm8f', 'fljdrtt', 'fmre7l9', 'fmspcex', 'fphcwq0', 'g3hmlew', 'givok1i', 'givp578'],
    "lyk25": ['dpirnlc', 'dpkib9q', 'e1vstxd', 'e1vv6x1', 'ewr7ls3', 'ewr7msm', 'fixjxxs', 'fixlxvy', 'fmre7l9', 'fmspcex'],
    "sqt2": ['d3efg03', 'd3f07tv', 'g0fwpzc', 'g0ggz8e', 'g8nyz4f', 'g8nzx3m', 'h4i75b9', 'h4i79lc', 'h6ikmzc', 'h6ime20'],
    "cd326": ['djh1bt1', 'djh2v24', 'dmlny27', 'dmlqnzz', 'f83akyz', 'f83hb2k', 'h28lknz', 'h28ltfw', 'i4rgtqf', 'i4rh1id'],
    "tg352": ['dm8exht', 'dm8qu78', 'dvisfl0', 'dvivs9p', 'dyx2jn4', 'dyx3au1', 'gvb1ekl', 'gvdnrrb', 'i4rgtqf', 'i4rh1id'],
}

# all_convo_ids = []
# for k,v in human_map1.items():
#     all_convo_ids.extend(v)
# all_convo_ids = list(set(all_convo_ids))
# len(all_convo_ids)

all_convo_ids = []
# # collect from round 1
if round_n == 1:
    for i in range(len(answers_list_1)):
        for j in range(len(answers_list_1[i])):
            all_convo_ids.append(answers_list_1[i][j]['id'])
# collect from round 2
elif round_n == 2:
    for i in range(len(answers_list_2)):
        for j in range(len(answers_list_2[i])):
            all_convo_ids.append(answers_list_2[i][j]['id'])
all_convo_ids = list(set(all_convo_ids))
print(f"[info] total unique conversation ids: {len(all_convo_ids)}")
print(all_convo_ids)

[info] total unique conversation ids: 84
['givp578', 'e1vstxd', 'f1353ra', 'ewr7ls3', 'dmlny27', 'dg7wmdb', 'flixm8f', 'f13u0n2', 'ggwdbsa', 'd3f07tv', 'isz5n6g', 'givok1i', 'dyx3au1', 'g1sm9mt', 'f0dryv4', 'doi44j8', 'g2998qp', 'dpirnlc', 'h2h7yl0', 'dg7x3eu', 'ej6jusi', 'ikzb571', 'e1vv6x1', 'ifkoazb', 'dpkib9q', 'g1q3upb', 'dydawov', 'g1q1973', 'dm8exht', 'fphcwq0', 'gmxqrdz', 'dmlqnzz', 'dyd7cfv', 'i4rgtqf', 'doi3vuz', 'd0fyu9x', 'f014doo', 'dm8qu78', 'gmxoyuu', 'fmre7l9', 'ii3cpve', 'ii4ivim', 'e9mynuy', 'djh1bt1', 'f83hb2k', 'ewtowqu', 'dbnxbzn', 'fixjxxs', 'dvivs9p', 'ej6ollb', 'g1spg0k', 'ggwei5y', 'd0fzp40', 'gvb1ekl', 'djh2v24', 'dvisfl0', 'ewtjxp2', 'dyx2jn4', 'ewr7msm', 'd3efg03', 'fixlxvy', 'fegw71k', 'ikzcblg', 'ffpk80c', 'fmspcex', 'f00nxjs', 'gvdnrrb', 'h28ltfw', 'f0dsbw4', 'dbnr5nd', 'ikpw1ol', 'ffpj88q', 'i4rh1id', 'e9mybxp', 'isz6a7m', 'ifknq44', 'g297nn7', 'fegwvia', 'ikpxahv', 'f83akyz', 'h2hauaj', 'h28lknz', 'g3hmlew', 'fljdrtt']


In [20]:
corpus = Corpus(filename=download('conversations-gone-awry-cmv-corpus-large'))
corpus.filter_conversations_by(lambda convo: convo.id in all_convo_ids)

Dataset already exists at /reef/lyk25/ConvoKit/examples/forecaster/conversations-gone-awry-cmv-corpus-large


In [21]:
# we want all utterances to have a human_guesses meta field
for convo in corpus.iter_conversations():
    for utt in convo.get_chronological_utterance_list():
        utt.add_meta('human_guesses', [])

In [ ]:
if round_n == 1:
    for i, participant_sessions in enumerate(answers_list_1):
        participant_id = names_list_1[i]
        print(participant_id)
        for session in participant_sessions:
            convo_id = session['id']
            convo = corpus.get_conversation(convo_id)
            utts = convo.get_chronological_utterance_list()
            actions = session.get('actions', [])

            for action_idx, action in enumerate(actions):
                if action.get('guess') is True:
                    utt = utts[action_idx]
                    # get current guesses (returns deep copy if exists, or empty list if not)
                    # create a new list to avoid mutating the copy
                    current_guesses = list(utt.meta.get('human_guesses', []))
                    # append new participant and set back
                    current_guesses.append(participant_id)
                    utt.add_meta('human_guesses', current_guesses)
elif round_n == 2:
    for i, participant_sessions in enumerate(answers_list_2):
        participant_id = names_list_2[i]
        print(participant_id)
        for session in participant_sessions:
            convo_id = session['id']
            convo = corpus.get_conversation(convo_id)
            utts = convo.get_chronological_utterance_list()
            actions = session.get('actions', [])

            for action_idx, action in enumerate(actions):
                if action.get('guess') is True:
                    utt = utts[action_idx]
                    # get current guesses (returns deep copy if exists, or empty list if not)
                    # create a new list to avoid mutating the copy
                    current_guesses = list(utt.meta.get('human_guesses', []))
                    # append new participant and set back
                    current_guesses.append(participant_id)
                    utt.add_meta('human_guesses', current_guesses)

vn72
nac86
sj597
ex36
kz88
LJL2
lyk25
cd326
tg352


In [23]:
# initialize model prediction metadata fields for each utterance
for convo in corpus.iter_conversations():
    for utt in convo.get_chronological_utterance_list():
        utt.add_meta('model_forecast_probs', {})  # will store {seed: prob}
        utt.add_meta('model_forecasts', {})  # will store {seed: binary_forecast}

In [ ]:
# copy model predictions from the corpora loaded above
loaded_seed_nums = sorted(
    int(corpus_name.rsplit("-", 1)[-1])
    for corpus_name in corpora
    if corpus_name.startswith("test-seed-")
)

if not loaded_seed_nums:
    available = ", ".join(sorted(corpora))
    raise KeyError(f"no test seed corpora found; available corpora: {available}")

for seed_num in loaded_seed_nums:
    seed_key = f"test-seed-{seed_num}"
    print(f"[info] loading predictions from seed {seed_num}: {seed_key}")
    seed_corpus = corpora[seed_key]

    for convo in corpus.iter_conversations():
        seed_convo = seed_corpus.get_conversation(convo.id)
        main_utts = convo.get_chronological_utterance_list()
        seed_utts = seed_convo.get_chronological_utterance_list()

        for main_utt, seed_utt in zip(main_utts, seed_utts):
            forecast_prob = seed_utt.meta.get("forecast_prob")
            forecast = seed_utt.meta.get("forecast")

            if forecast_prob is not None:
                current_probs = dict(main_utt.meta.get("model_forecast_probs", {}))
                current_probs[f"seed_{seed_num}"] = forecast_prob
                main_utt.add_meta("model_forecast_probs", current_probs)

            if forecast is not None:
                current_forecasts = dict(main_utt.meta.get("model_forecasts", {}))
                current_forecasts[f"seed_{seed_num}"] = forecast
                main_utt.add_meta("model_forecasts", current_forecasts)

print("\n[pass] model predictions from all seeds added to corpus")


In [5]:
# horizon comparison between gemma and humans, using the same convention as
# performance_utils_wiki.calculate_current_performance:
#   - only count true positives (ground truth awry AND trigger fired)
#   - horizon = (len(utts) - first_trigger_index) - 1
#   - first trigger only (break after first positive)
#
# reports per-round results for humans (round 1 and round 2), pulling data
# directly from the game sqlite so it works regardless of which round_n the
# rest of the notebook was run under. ljl2 is excluded.

import sqlite3
import json as _json
from collections import defaultdict
import numpy as np

def _horizon_mean(horizons):
    # matches performance_utils_wiki: h = mean(horizons) - 1
    if len(horizons) == 0:
        return float('nan'), 0
    return float(np.mean(horizons)) - 1, len(horizons)

# always excluded (staff/test accounts). ljl2 is a legitimate late submitter
# in the round 2_b window, not an exclusion.
base_excluded = {'yc2727', 'sqt2'}
round1_excluded = set(base_excluded)
round2_excluded = set(base_excluded)

# gemma: per-seed first-trigger horizons on TP convos only (ground truth awry)
gemma_horizons_by_seed = defaultdict(list)
for convo in corpus.iter_conversations():
    if not convo.meta.get('has_removed_comment', False):
        continue  # skip non-awry convos; horizon is only meaningful on TPs
    utts = convo.get_chronological_utterance_list()
    for seed_num in range(1, 6):
        for i, utt in enumerate(utts):
            if utt.meta.get('model_forecasts', {}).get(f'seed_{seed_num}') == 1:
                gemma_horizons_by_seed[seed_num].append(len(utts) - i)
                break

print('gemma horizon (TPs only, mean - 1):')
gemma_per_seed_h = []
for seed_num in sorted(gemma_horizons_by_seed.keys()):
    h, n = _horizon_mean(gemma_horizons_by_seed[seed_num])
    gemma_per_seed_h.append(h)
    print(f'  seed {seed_num}: n={n}, h={h:.4f}')
gemma_h_mean_of_seeds = float(np.mean(gemma_per_seed_h)) if gemma_per_seed_h else float('nan')
all_gemma_vals = [v for vs in gemma_horizons_by_seed.values() for v in vs]
gemma_h_pooled, gemma_n_pooled = _horizon_mean(all_gemma_vals)
print(f'  mean over seeds: h={gemma_h_mean_of_seeds:.4f}')
print(f'  pooled: n={gemma_n_pooled}, h={gemma_h_pooled:.4f}')

# pull round-specific human guesses directly from the sqlite db
db_path = '/reef/sqt2/cga-eval/human/game_db.sqlite'
_conn = sqlite3.connect(db_path)

def _load_round_answers(round_n):
    # returns list of (participant_id, sessions) for the given round
    entries = []
    if round_n == 1:
        excluded = round1_excluded
        t_start, t_end = 1730395000000, 1730397199000
        for row in _conn.execute('SELECT * FROM results'):
            if row[0] in excluded:
                continue
            answers = _json.loads(row[1])
            if answers and answers[0]['start_time'] > t_start and answers[0]['start_time'] < t_end:
                entries.append((row[0], answers))
    elif round_n == 2:
        excluded = round2_excluded
        t_start_a, t_end_a = 1731002910000, 1731016180000
        t_start_b, t_end_b = 1732212720000, 1740000000000
        for row in _conn.execute('SELECT * FROM results'):
            name = row[0]
            if name in excluded:
                continue
            answers = _json.loads(row[1])
            if not answers:
                continue
            st = answers[0]['start_time']
            in_a = t_start_a < st < t_end_a
            # the 2_b late window is only valid for ljl2 (matches cell 7)
            in_b = (name.lower() == 'ljl2') and (t_start_b < st < t_end_b)
            if in_a or in_b:
                entries.append((name, answers))
    return entries

def _unique_convo_ids(entries):
    # unique convo ids seen across all included players' sessions
    ids = set()
    for _, sessions in entries:
        for session in sessions:
            cid = session.get('id')
            if cid is not None:
                ids.add(cid)
    return ids

def _compute_human_horizons(entries):
    # returns dict: player_id -> list of horizons (non-awry convos only, first guess per convo)
    by_player = defaultdict(list)
    for participant_id, sessions in entries:
        for session in sessions:
            convo_id = session.get('id')
            if convo_id is None:
                continue
            try:
                convo = corpus.get_conversation(convo_id)
            except KeyError:
                # convo not in the loaded corpus (e.g., different round loaded)
                continue
            if not convo.meta.get('has_removed_comment', False):
                continue  # skip non-awry convos; horizon only counted on TPs
            utts = convo.get_chronological_utterance_list()
            for action_idx, action in enumerate(session.get('actions', [])):
                if action.get('guess') is True:
                    if action_idx < len(utts):
                        by_player[participant_id].append(len(utts) - action_idx)
                    break  # only first guess per (player, convo)
    return by_player

def _print_human_horizons(label, by_player):
    print(f'human horizon ({label}, TPs only, mean - 1):')
    player_h = []
    for player, horizons in by_player.items():
        h, n = _horizon_mean(horizons)
        player_h.append(h)
        print(f'  player {player}: n={n}, h={h:.4f}')
    mean_of_players = float(np.mean(player_h)) if player_h else float('nan')
    all_vals = [v for vs in by_player.values() for v in vs]
    h_pooled, n_pooled = _horizon_mean(all_vals)
    print(f'  mean over players: h={mean_of_players:.4f}')
    print(f'  pooled: n={n_pooled}, h={h_pooled:.4f}')
    return mean_of_players, h_pooled

EXPECTED_N_CONVOS = 84
round1_entries = _load_round_answers(1)
round1_convo_ids = _unique_convo_ids(round1_entries)
round1_by_player = _compute_human_horizons(round1_entries)
r1_mean, r1_pooled = _print_human_horizons('round 1', round1_by_player)

round2_entries = _load_round_answers(2)
round2_convo_ids = _unique_convo_ids(round2_entries)
round2_by_player = _compute_human_horizons(round2_entries)
r2_mean, r2_pooled = _print_human_horizons('round 2', round2_by_player)

_conn.close()

print()
print('summary (comparable to h in performance_utils_wiki):')
print(f'  gemma h (mean over seeds):   {gemma_h_mean_of_seeds:.4f}')
print(f'  round1 h (mean over players): {r1_mean:.4f}')
print(f'  round2 h (mean over players): {r2_mean:.4f}')


gemma horizon (TPs only, mean - 1):
  mean over seeds: h=nan
  pooled: n=0, h=nan
human horizon (round 1, TPs only, mean - 1):
  player vn72: n=1, h=6.0000
  player nac86: n=2, h=0.5000
  player sj597: n=4, h=2.2500
  player ex36: n=1, h=2.0000
  player kz88: n=2, h=2.5000
  player LJL2: n=4, h=1.5000
  player lyk25: n=4, h=2.5000
  player cd326: n=2, h=1.5000
  player tg352: n=2, h=5.0000
  mean over players: h=2.6389
  pooled: n=22, h=2.3636
human horizon (round 2, TPs only, mean - 1):
  player sj597: n=4, h=1.2500
  player nac86: n=2, h=4.0000
  player ex36: n=2, h=0.5000
  player vn72: n=3, h=1.6667
  player lyk25: n=4, h=3.0000
  player kz88: n=1, h=3.0000
  player cd326: n=4, h=3.2500
  player tg352: n=3, h=2.0000
  player LJL2: n=2, h=0.5000
  mean over players: h=2.1296
  pooled: n=25, h=2.1600

summary (comparable to h in performance_utils_wiki):
  gemma h (mean over seeds):   nan
  round1 h (mean over players): 2.6389
  round2 h (mean over players): 2.1296


In [6]:
from collections import defaultdict
import numpy as np
def _compute_metrics(tp, fp, tn, fn):
    total = tp + fp + tn + fn
    accuracy = (tp + tn) / total if total > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'f1': f1, 'fpr': fpr, 'specificity': specificity, 'fnr': fnr,
    }
def _gt(convo):
    return bool(convo.meta.get('has_removed_comment', False))
# gemma benchmark, per seed, then mean and std
gemma_per_seed = []
for seed_num in range(1, 6):
    tp = fp = tn = fn = 0
    for convo in corpus.iter_conversations():
        utts = convo.get_chronological_utterance_list()
        pred = any(
            utt.meta.get('model_forecasts', {}).get(f'seed_{seed_num}') == 1
            for utt in utts
        )
        truth = _gt(convo)
        if pred and truth: tp += 1
        elif pred and not truth: fp += 1
        elif not pred and not truth: tn += 1
        elif not pred and truth: fn += 1
    gemma_per_seed.append(_compute_metrics(tp, fp, tn, fn))
gemma_mean = {k: float(np.mean([m[k] for m in gemma_per_seed])) for k in gemma_per_seed[0]}
gemma_std = {k: float(np.std([m[k] for m in gemma_per_seed], ddof=1)) for k in gemma_per_seed[0]}
def _build_per_player_preds(entries):
    # returns dict[player_id] -> dict[convo_id] -> 0/1
    preds = defaultdict(dict)
    for player, sessions in entries:
        for s in sessions:
            cid = s.get('id')
            if cid is None:
                continue
            try:
                corpus.get_conversation(cid)
            except KeyError:
                continue
            actions = s.get('actions', [])
            preds[player][cid] = 1 if any(a.get('guess') is True for a in actions) else 0
    return preds
def _aggregate_any(preds):
    # per-convo prediction = 1 if any player who saw it guessed
    by_convo = defaultdict(list)
    for player, convo_preds in preds.items():
        for cid, p in convo_preds.items():
            by_convo[cid].append(p)
    tp = fp = tn = fn = 0
    for cid, plist in by_convo.items():
        try:
            convo = corpus.get_conversation(cid)
        except KeyError:
            continue
        pred = 1 if any(plist) else 0
        truth = 1 if _gt(convo) else 0
        if pred and truth: tp += 1
        elif pred and not truth: fp += 1
        elif not pred and not truth: tn += 1
        elif not pred and truth: fn += 1
    return _compute_metrics(tp, fp, tn, fn), len(by_convo)
def _per_player_metrics(preds):
    rows = {}
    for player, convo_preds in preds.items():
        tp = fp = tn = fn = 0
        for cid, p in convo_preds.items():
            try:
                convo = corpus.get_conversation(cid)
            except KeyError:
                continue
            truth = 1 if _gt(convo) else 0
            if p and truth: tp += 1
            elif p and not truth: fp += 1
            elif not p and not truth: tn += 1
            elif not p and truth: fn += 1
        rows[player] = _compute_metrics(tp, fp, tn, fn)
    return rows
# round 1 and round 2 humans
round1_preds = _build_per_player_preds(round1_entries)
round1_agg, round1_n_convos = _aggregate_any(round1_preds)
round1_per_player = _per_player_metrics(round1_preds)
round2_preds = _build_per_player_preds(round2_entries)
round2_agg, round2_n_convos = _aggregate_any(round2_preds)
round2_per_player = _per_player_metrics(round2_preds)
# metric mean over players (a different aggregation view)
def _mean_over_players(per_player):
    keys = ['accuracy', 'precision', 'recall', 'f1', 'fpr', 'specificity', 'fnr']
    return {k: float(np.mean([m[k] for m in per_player.values()])) for k in keys}
round1_mean_over_players = _mean_over_players(round1_per_player)
round2_mean_over_players = _mean_over_players(round2_per_player)
# printing
metric_order = ['accuracy', 'precision', 'recall', 'f1', 'fpr', 'specificity', 'fnr']
def _print_per_player(label, per_player):
    print(f'{label} - per-player metrics:')
    header = f"  {'player':<10}" + "".join(f"{m:>12}" for m in metric_order) + f"{'n_convos':>10}"
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for player in sorted(per_player):
        m = per_player[player]
        n = m['tp'] + m['fp'] + m['tn'] + m['fn']
        row = f"  {player:<10}" + "".join(f"{m[k]:>12.4f}" for k in metric_order) + f"{n:>10d}"
        print(row)


In [29]:
pm = '\u00b1'
metric_order = ['accuracy', 'precision', 'recall', 'f1', 'fpr', 'specificity', 'fnr']

def _print_per_player(label, per_player):
    print(f'{label} - per-player metrics:')
    header = f"  {'player':<10}" + "".join(f"{m:>12}" for m in metric_order) + f"{'n_convos':>10}"
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for player in sorted(per_player):
        m = per_player[player]
        n = m['tp'] + m['fp'] + m['tn'] + m['fn']
        row = f"  {player:<10}" + "".join(f"{m[k]:>12.4f}" for k in metric_order) + f"{n:>10d}"
        print(row)

_print_per_player('round 1', round1_per_player)
print()
_print_per_player('round 2', round2_per_player)

print()
print(f'aggregate benchmark (gemma: mean{pm}std over 5 seeds; humans: any-player rule + mean over players)')

header = f"{'group':<36}" + "".join(f"{m:>14}" for m in metric_order)
print(header)
print('-' * len(header))

def _fmt_mean_std(mean_dict, std_dict):
    return "".join(f"  {mean_dict[k]:.3f}{pm}{std_dict[k]:.3f}" for k in metric_order)

def _fmt_mean(mean_dict):
    return "".join(f"{mean_dict[k]:>14.4f}" for k in metric_order)

gemma_label = f'gemma (mean{pm}std over seeds)'
print(f"{gemma_label:<36}" + _fmt_mean_std(gemma_mean, gemma_std))
print(f"{'round1 humans (mean over players)':<36}" + _fmt_mean(round1_mean_over_players))
print(f"{'round2 humans (mean over players)':<36}" + _fmt_mean(round2_mean_over_players))

print()
print(f'round 1 unique convos: {round1_n_convos}')
print(f'round 2 unique convos: {round2_n_convos}')

round 1 - per-player metrics:
  player        accuracy   precision      recall          f1         fpr specificity         fnr  n_convos
  --------------------------------------------------------------------------------------------------------
  LJL2            0.8000      0.8000      0.8000      0.8000      0.2000      0.8000      0.2000        10
  cd326           0.6000      0.6667      0.4000      0.5000      0.2000      0.8000      0.6000        10
  ex36            0.4000      0.3333      0.2000      0.2500      0.4000      0.6000      0.8000        10
  kz88            0.7000      1.0000      0.4000      0.5714      0.0000      1.0000      0.6000        10
  lyk25           0.7000      0.6667      0.8000      0.7273      0.4000      0.6000      0.2000        10
  nac86           0.7000      1.0000      0.4000      0.5714      0.0000      1.0000      0.6000        10
  sj597           0.8000      0.8000      0.8000      0.8000      0.2000      0.8000      0.2000        10
  tg352

## Validation of forecast probability decrease

In [1]:
import json
from pathlib import Path
from convokit import Corpus

corpus_base = Path("/reef/lyk25/dynamic_training/game_analysis/corpi/test")
config_base = Path("/reef/sqt2/FinalAAO/cga-cmv-large/google/gemma-2-9b-it")

corpus_dirs = sorted(
    corpus_dir
    for corpus_dir in corpus_base.rglob("*")
    if corpus_dir.is_dir() and (corpus_dir / "index.json").exists()
)

if not corpus_dirs:
    raise FileNotFoundError(f"no convokit corpora found under {corpus_base}")


def _test_seed_key(corpus_dir):
    parent_name = corpus_dir.parent.name
    if parent_name.startswith("test-seed-"):
        return parent_name
    seed_idx = corpus_dir.name.rsplit("-", 1)[-1]
    return f"test-seed-{seed_idx}"


corpus_path_by_key = {}
for corpus_dir in corpus_dirs:
    key = _test_seed_key(corpus_dir)
    corpus_path_by_key[key] = corpus_dir


def is_calm_sim_forecast(forecast):
    if isinstance(forecast, str):
        forecast = forecast.strip().lower()
        if forecast in {"0", "false", "calm"}:
            return True
        if forecast in {"1", "true", "awry"}:
            return False
    return int(forecast) == 0


def count_decreases_at_indices(corpus, get_indices):
    total = 0
    reduction = 0

    for convo in corpus.iter_conversations():
        utts = convo.get_chronological_utterance_list()

        for i in get_indices(utts):
            if i + 1 >= len(utts):
                continue

            f1 = utts[i].meta["forecast_prob"]
            f2 = utts[i + 1].meta["forecast_prob"]
            total += 1

            if f1 > f2:
                reduction += 1

    return reduction, total


def current_trigger_indices(utts, pred_threshold):
    return [
        i
        for i, utt in enumerate(utts)
        if utt.meta["forecast_prob"] > pred_threshold
    ]


def delayed_indices_from_sim_forecasts(utts, pred_threshold, k):
    delayed = []

    for i, utt in enumerate(utts):
        if utt.meta["forecast_prob"] <= pred_threshold:
            continue

        sim_forecasts = utt.meta["sim_replies_forecasts"]
        calm_sim_replies = sum(
            1 for forecast in sim_forecasts if is_calm_sim_forecast(forecast)
        )

        if calm_sim_replies > k:
            delayed.append(i)

    return delayed


current_reduction = 0
current_total = 0
delayed_reduction = 0
delayed_total = 0

for seed in range(1, 6):
    seed_key = f"test-seed-{seed}"
    corpus = Corpus(filename=str(corpus_path_by_key[seed_key]))

    with open(config_base / f"seed-{seed}" / "dev_config.json") as f:
        best_threshold = json.load(f)["best_threshold"]

    r, t = count_decreases_at_indices(
        corpus,
        lambda utts: current_trigger_indices(utts, best_threshold),
    )
    current_reduction += r
    current_total += t

    r, t = count_decreases_at_indices(
        corpus,
        lambda utts: delayed_indices_from_sim_forecasts(utts, best_threshold, k=7),
    )
    delayed_reduction += r
    delayed_total += t

pooled_trigger_all = current_reduction / current_total
pooled_delayed_all = delayed_reduction / delayed_total

print(f"pooled trigger_all current perf, best threshold per seed: {pooled_trigger_all} ({current_reduction}/{current_total})")
print(f"pooled delayed_all best threshold per seed, k=7: {pooled_delayed_all} ({delayed_reduction}/{delayed_total})")

pooled trigger_all current perf, best threshold per seed: 0.5532229185317815 (12359/22340)
pooled delayed_all best threshold per seed, k=7: 0.8353165159447882 (3510/4202)


## Calculating oracle threshold

In [9]:
# corpora comes from the decisionpolicy-demo download cell; use only test-seed-<n> entries.
import re

if "corpora" not in globals() or not corpora:
    raise RuntimeError(
        "corpora is empty: run the download cell above first."
    )

_prev_keys = tuple(sorted(corpora))
_test_seed_re = re.compile(r"^test-seed-(\d+)$")

corpora = {
    k: corpora[k]
    for k in sorted(
        (k for k in _prev_keys if _test_seed_re.match(k)),
        key=lambda k: int(_test_seed_re.match(k).group(1)),
    )
}

if not corpora:
    raise RuntimeError(
        "no test-seed-<n> corpora after filter; had keys: "
        + ", ".join(_prev_keys)
    )

print(
    f"[info] using {len(corpora)} test-seed corpora: "
    + ", ".join(corpora)
)

[info] using 5 test-seed corpora: test-seed-1, test-seed-2, test-seed-3, test-seed-4, test-seed-5


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

seed_best_thresholds = {
    "test-seed-1": 0.5926666259765625,
    "test-seed-2": 0.622459352016449,
    "test-seed-3": 0.6513549089431763,
    "test-seed-4": 0.6513549089431763,
    "test-seed-5": 0.6513549089431763,
}

mean_best_threshold = float(np.mean(list(seed_best_thresholds.values())))
search_radius = 0.15
num_thresholds = 400
baseline_thresholds = np.linspace(
    mean_best_threshold - search_radius,
    mean_best_threshold + search_radius,
    num_thresholds,
)

print(f"[info] per-seed best thresholds: {seed_best_thresholds}")
print(f"[info] mean best threshold: {mean_best_threshold:.6f}")
print(f"[info] generating baseline roc curve with {len(baseline_thresholds)} thresholds in [{baseline_thresholds[0]:.6f}, {baseline_thresholds[-1]:.6f}]")

# k values to test for our method
k_values = list(range(1, 11))  # 1 to 10
print(f"[info] testing k values: {k_values}")


[info] per-seed best thresholds: {'test-seed-1': 0.5926666259765625, 'test-seed-2': 0.622459352016449, 'test-seed-3': 0.6513549089431763, 'test-seed-4': 0.6513549089431763, 'test-seed-5': 0.6513549089431763}
[info] mean best threshold: 0.633838
[info] generating baseline roc curve with 400 thresholds in [0.483838, 0.783838]
[info] testing k values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [10]:
import numpy as np
import pandas as pd
from convokit.decisionpolicy import ThresholdDecisionPolicy
from convokit.forecaster.forecaster import ContextTuple


def _cached_forecast_score(context):
    meta = getattr(context.current_utterance, "meta", {}) or {}
    if "forecast_prob" not in meta:
        raise KeyError(f"missing forecast_prob for utterance {context.current_utterance.id}")
    return meta["forecast_prob"]


def _threshold_policy_performance(corpus, policy):
    tp = fp = tn = fn = 0
    horizons = []

    for convo in corpus.iter_conversations():
        utts = convo.get_chronological_utterance_list()
        pred = 0
        first_trigger_idx = None

        for utt_idx, utt in enumerate(utts):
            context = ContextTuple(
                context=utts[: utt_idx + 1],
                current_utterance=utt,
                future_context=utts[utt_idx + 1 :],
                conversation_id=convo.id,
            )
            _, utt_pred = policy.decide(context, _cached_forecast_score)
            if int(utt_pred) == 1:
                pred = 1
                first_trigger_idx = utt_idx
                break

        truth = bool(convo.meta.get("has_removed_comment", False))
        if pred and truth:
            tp += 1
            horizons.append(len(utts) - first_trigger_idx)
        elif pred and not truth:
            fp += 1
        elif not pred and not truth:
            tn += 1
        else:
            fn += 1

    h = float(np.mean(horizons)) - 1 if horizons else float("nan")
    return {
        "confusion_matrix": {"TP": tp, "FP": fp, "TN": tn, "FN": fn},
        "h": h,
    }


# step 1: compute baseline roc curve with ThresholdDecisionPolicy
print("[info] computing baseline roc curves for all seeds...")
all_baseline_results = {}

for seed_name, corpus in corpora.items():
    print(f"\n[info] processing {seed_name}...")
    baseline_results = []

    for i, threshold in enumerate(baseline_thresholds):
        if i % 20 == 0:
            print(f"[info] {seed_name} baseline progress: {i+1}/{len(baseline_thresholds)}")

        policy = ThresholdDecisionPolicy(threshold=threshold)
        results = _threshold_policy_performance(corpus, policy)
        tp = results["confusion_matrix"]["TP"]
        fp = results["confusion_matrix"]["FP"]
        tn = results["confusion_matrix"]["TN"]
        fn = results["confusion_matrix"]["FN"]

        # calculate tpr, fpr, accuracy, precision, recall, f1
        # recall == tpr; kept as a separate field for downstream readability
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        total = tp + fp + tn + fn
        accuracy = (tp + tn) / total if total > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tpr
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        baseline_results.append({
            "seed": seed_name,
            "threshold": threshold,
            "tpr": tpr,
            "fpr": fpr,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "h": results.get("h", float("nan")),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        })

    all_baseline_results[seed_name] = pd.DataFrame(baseline_results)
    print(f"[pass] {seed_name} baseline complete - {len(all_baseline_results[seed_name])} points")

print(f"\n[pass] all baseline roc curves computed for {len(all_baseline_results)} seeds")


[info] computing baseline roc curves for all seeds...

[info] processing test-son-seed-1...
[info] test-son-seed-1 baseline progress: 1/400
[info] test-son-seed-1 baseline progress: 21/400
[info] test-son-seed-1 baseline progress: 41/400
[info] test-son-seed-1 baseline progress: 61/400
[info] test-son-seed-1 baseline progress: 81/400
[info] test-son-seed-1 baseline progress: 101/400
[info] test-son-seed-1 baseline progress: 121/400
[info] test-son-seed-1 baseline progress: 141/400
[info] test-son-seed-1 baseline progress: 161/400
[info] test-son-seed-1 baseline progress: 181/400
[info] test-son-seed-1 baseline progress: 201/400
[info] test-son-seed-1 baseline progress: 221/400
[info] test-son-seed-1 baseline progress: 241/400
[info] test-son-seed-1 baseline progress: 261/400
[info] test-son-seed-1 baseline progress: 281/400
[info] test-son-seed-1 baseline progress: 301/400
[info] test-son-seed-1 baseline progress: 321/400
[info] test-son-seed-1 baseline progress: 341/400
[info] test-so

In [11]:
import pickle

with open('all_baseline_results.pkl', 'wb') as f:
    pickle.dump(all_baseline_results, f)
print("[info] all_baseline_results has been pickled to all_baseline_results.pkl")

[info] all_baseline_results has been pickled to all_baseline_results.pkl


In [8]:
import json

import numpy as np
import pandas as pd
from convokit.decisionpolicy import DeferralDecisionPolicy
from convokit.forecaster.forecaster import ContextTuple


def _cached_forecast_score(context):
    meta = getattr(context.current_utterance, "meta", {}) or {}
    if "forecast_prob" not in meta:
        raise KeyError(f"missing forecast_prob for utterance {context.current_utterance.id}")
    return meta["forecast_prob"]


def _deferral_policy_performance(corpus, policy):
    tp = fp = tn = fn = 0
    horizons = []

    for convo in corpus.iter_conversations():
        utts = convo.get_chronological_utterance_list()
        pred = 0
        first_trigger_idx = None

        for utt_idx, utt in enumerate(utts):
            context = ContextTuple(
                context=utts[: utt_idx + 1],
                current_utterance=utt,
                future_context=utts[utt_idx + 1 :],
                conversation_id=convo.id,
            )
            result = policy.decide(context, _cached_forecast_score)
            utt_pred = int(result[1])
            if utt_pred == 1:
                pred = 1
                first_trigger_idx = utt_idx
                break

        truth = bool(convo.meta.get("has_removed_comment", False))
        if pred and truth:
            tp += 1
            horizons.append(len(utts) - first_trigger_idx)
        elif pred and not truth:
            fp += 1
        elif not pred and not truth:
            tn += 1
        else:
            fn += 1

    h = float(np.mean(horizons)) - 1 if horizons else float("nan")
    return {
        "confusion_matrix": {"TP": tp, "FP": fp, "TN": tn, "FN": fn},
        "h": h,
    }


all_k_results = {}

for seed_name, corpus in corpora.items():
    print(f"\n[info] processing {seed_name}...")
    k_results = []

    pruned_seed_name = seed_name[len(seed_name) - 6 :]
    print(f"[info] config seed: {pruned_seed_name}")

    with open(f"/reef/sqt2/FinalAAO/cga-cmv-large/google/gemma-2-9b-it/{pruned_seed_name}/dev_config.json", "r") as f:
        dev_config = json.load(f)

    k_method_threshold = dev_config["best_threshold"]
    print("[info] k method threshold", k_method_threshold)

    for k in k_values:
        print(f"[info] {seed_name} computing k={k}...")
        policy = DeferralDecisionPolicy(
            simulator=None,
            threshold=k_method_threshold,
            tau=k,
            num_simulations=10,
            store_simulations=False,
            simulated_reply_attribute_name="sim_replies",
            sim_replies_forecast_probs_attribute_name="sim_replies_forecast_probs",
            reuse_cached_simulations=True,
        )
        results = _deferral_policy_performance(corpus, policy)
        tp = results["confusion_matrix"]["TP"]
        fp = results["confusion_matrix"]["FP"]
        tn = results["confusion_matrix"]["TN"]
        fn = results["confusion_matrix"]["FN"]

        # calculate tpr, fpr, accuracy, precision, recall, f1
        # recall == tpr; kept as a separate field for downstream readability
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        total = tp + fp + tn + fn
        accuracy = (tp + tn) / total if total > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tpr
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        k_results.append({
            "seed": seed_name,
            "k": k,
            "threshold": k_method_threshold,
            "tpr": tpr,
            "fpr": fpr,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "h": results.get("h", float("nan")),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        })

    all_k_results[seed_name] = pd.DataFrame(k_results)
    print(f"[pass] {seed_name} k-method complete - {len(all_k_results[seed_name])} points")

print(f"\n[pass] all k-method results computed for {len(all_k_results)} seeds")



[info] processing test-son-seed-1...
[info] config seed: seed-1
[info] k method threshold 0.5926666259765625
[info] test-son-seed-1 computing k=1...
[info] test-son-seed-1 computing k=2...
[info] test-son-seed-1 computing k=3...
[info] test-son-seed-1 computing k=4...
[info] test-son-seed-1 computing k=5...
[info] test-son-seed-1 computing k=6...
[info] test-son-seed-1 computing k=7...
[info] test-son-seed-1 computing k=8...
[info] test-son-seed-1 computing k=9...
[info] test-son-seed-1 computing k=10...
[pass] test-son-seed-1 k-method complete - 10 points

[info] processing test-son-seed-2...
[info] config seed: seed-2
[info] k method threshold 0.622459352016449
[info] test-son-seed-2 computing k=1...
[info] test-son-seed-2 computing k=2...
[info] test-son-seed-2 computing k=3...
[info] test-son-seed-2 computing k=4...
[info] test-son-seed-2 computing k=5...
[info] test-son-seed-2 computing k=6...
[info] test-son-seed-2 computing k=7...
[info] test-son-seed-2 computing k=8...
[info] 

In [13]:
# step 3: for each k's FPR, find the closest baseline FPR (per seed)
print("[info] matching fprs for all seeds...")
all_matched_results = {}

for seed_name in corpora.keys():
    print(f"\n[info] matching fprs for {seed_name}...")
    matched_results = []
    
    k_df = all_k_results[seed_name]
    baseline_df = all_baseline_results[seed_name]
    
    for _, k_row in k_df.iterrows():
        k_fpr = k_row['fpr']
        k_tpr = k_row['tpr']
        k_val = k_row['k']
        
        # find closest baseline fpr
        fpr_diffs = np.abs(baseline_df['fpr'] - k_fpr)
        closest_idx = fpr_diffs.idxmin()
        baseline_match = baseline_df.iloc[closest_idx]
        
        matched_results.append({
            'seed': seed_name,
            'k': k_val,
            'k_fpr': k_fpr,
            'k_tpr': k_tpr,
            'baseline_fpr': baseline_match['fpr'],
            'baseline_tpr': baseline_match['tpr'],
            'baseline_accuracy': baseline_match['accuracy'],
            'baseline_precision': baseline_match['precision'],
            'baseline_recall': baseline_match['recall'],
            'baseline_f1': baseline_match['f1'],
            'baseline_h': baseline_match['h'],
            'baseline_threshold': baseline_match['threshold'],
            'fpr_diff': abs(k_fpr - baseline_match['fpr']),
            'tpr_improvement': k_tpr - baseline_match['tpr']
        })
    
    all_matched_results[seed_name] = pd.DataFrame(matched_results)
    print(f"[PASS] {seed_name} matched {len(all_matched_results[seed_name])} fpr points")

print(f"\n[PASS] all matching complete for {len(all_matched_results)} seeds")


[info] matching fprs for all seeds...

[info] matching fprs for test-son-seed-1...
[PASS] test-son-seed-1 matched 10 fpr points

[info] matching fprs for test-son-seed-2...
[PASS] test-son-seed-2 matched 10 fpr points

[info] matching fprs for test-son-seed-3...
[PASS] test-son-seed-3 matched 10 fpr points

[info] matching fprs for test-son-seed-4...
[PASS] test-son-seed-4 matched 10 fpr points

[info] matching fprs for test-son-seed-5...
[PASS] test-son-seed-5 matched 10 fpr points

[PASS] all matching complete for 5 seeds


In [18]:
# average matched-baseline (oracle) metrics for tau=7, across all seeds
target_tau = 7
matched_long = pd.concat(
    [df.assign(seed=seed_name) for seed_name, df in all_matched_results.items()],
    ignore_index=True,
)
matched_long = matched_long[matched_long["k"] == target_tau].copy()

if matched_long.empty:
    raise ValueError(f"no matched results found for tau={target_tau}")

oracle_cols = [
    "baseline_accuracy",
    "baseline_precision",
    "baseline_recall",
    "baseline_f1",
    "baseline_h",
    "fpr_diff",
    "tpr_improvement",
    "baseline_threshold",
    "baseline_fpr",
    "baseline_tpr",
]

oracle_stats_per_tau = (
    matched_long.groupby("k")[oracle_cols]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"k": "tau"})
)

# reformat columns to single-level, e.g. baseline_fpr_mean
oracle_stats_per_tau.columns = ["tau"] + [
    f"{col}_{stat}" for col in oracle_cols for stat in ["mean", "std"]
]

with pd.option_context(
    "display.float_format",
    "{:.4f}".format,
    "display.max_columns",
    None,
    "display.width",
    200,
):
    print("[info] matched fpr-oracle metrics for tau=7 "
          f"(mean ± std over seeds, n={matched_long['seed'].nunique()}):")
    print(oracle_stats_per_tau.to_string(index=False))

[info] matched fpr-oracle metrics for tau=7 (mean ± std over seeds, n=5):
 tau  baseline_accuracy_mean  baseline_accuracy_std  baseline_precision_mean  baseline_precision_std  baseline_recall_mean  baseline_recall_std  baseline_f1_mean  baseline_f1_std  baseline_h_mean  baseline_h_std  fpr_diff_mean  fpr_diff_std  tpr_improvement_mean  tpr_improvement_std  baseline_threshold_mean  baseline_threshold_std  baseline_fpr_mean  baseline_fpr_std  baseline_tpr_mean  baseline_tpr_std
   7                  0.7002                 0.0088                   0.7152                  0.0100                0.6677               0.0494            0.6896           0.0218           2.6965          0.1029         0.0036        0.0028                0.0155               0.0055                   0.6900                  0.0245             0.2674            0.0331             0.6677            0.0494
